This notebook contains the review questions for the midterm exam.

In [1]:
using SymPy

In [2]:
using JuMP
import DataFrames
import GLPK

In [3]:
using NLsolve

# 1. Taguchi quality control

Suppose a medicine to reduce fever is effective in 80% of the cases
after the first dosis and costs $1.

A temperature reading of more than two degrees above normal 
six hours after taking the first dosis requires a visit 
to the doctor which costs $10.

Clinical tests for a new fewer reducing medicine reported a success
rate of 95% after taking the first dosis.  

1. Under the same circumstances as above, is a cost of $2 for the
   new medicine justified?
        
   (*Hint:* compute the expected loss of having a fever for both medicines.)

2. What is a fair price for the new medicine, taking into account its higher quality?

At which price would you buy the new medicine?

## computing the expected loss

The expected loss for the first medicine:

$$
   0.80 ~\cdot~ \$1 + 0.2 \cdot ( \$1 + \$10 ) = \$3.0.
$$

For the new medicine:

$$
   0.95 ~\cdot~ \$2 + 0.05 \cdot ( \$2 + \$10 ) = \$2.5.
$$

The patient is better off with the new medicine.

## the fair price for the new medicine

For a fair price, we assume the same expected loss.

Let $x$ be the break even price, with both medicines the same loss.

$$
\begin{eqnarray*}
   3.0 & = & 0.95 x + 0.05(x + 10) \\
       & = & x + 0.5 \\
       & \Rightarrow & x = \$2.5.
\end{eqnarray*}
$$
If $x \leq \$2.5$, then we buy the medicine.
Otherwise not.

# 2. running a simulation

A man decides to participate in a 1,000 mile car race.

Because of the rough terrain and excessive speed,
tires have a life time normally distributed with an
average of 600 miles and standard deviation of 200 miles.

The man starts the race with 4 new tires and 4 spare tires.

Given the expected life span of the tires,
what are the odds to finish the 1,000 mile race?

1. Set up a simulation to decide the odds of to finish the race.
    
2. Run the simulation 10,000 times and give the probability of reaching the finish.

## a Julia function

In [4]:
"""
    Runs N simulations and returns the
number of times the finish was reached.
"""
function run(N::Int64)
    successes = 0
    for k=1:N
        tires = 600 .+ randn(8)*200
        life = tires[1:4]
        for j=1:4
            life = sort(life)
            life[1] = life[1] + tires[4+j]
        end
        if minimum(life) >= 1000
            successes = successes + 1
        end
    end
    return successes
end

run

In [5]:
println(run(10000)/10000)

0.3739


So, the probability of finishing the race is about .37

# 3. a recursion

Suppose we start with an empty savings account.

At the last day of every month $400 is deposited into the account.

After the first deposit, 20% of the balance of the savings account
is withdrawn at the first day of every month.

1. Set up the recursion relation for the balance $B(n)$
         at month $n$, after $n$ deposits.
    
2. Solve the recursion for $B(n)$.
        
3. In doing so, will we ever get rich?  Is there a limit to $B(n)$?

## defining and solving the recursion

$$
   B(n) = 0.8 B(n-1) + 400, B(0) = 0.
$$

We can solve this recursion by substitution.

$$
\begin{eqnarray*}
  B(n) & = & {\displaystyle \frac{4}{5} ~ B(n-1) + 400} \\
       & = & {\displaystyle \frac{4}{5} 
   \left( \frac{4}{5} ~ B(n-2) + 400 \right) + 400} \\
       & \vdots & \\
       & = & {\displaystyle \left( \frac{4}{5} \right)^n B(0)
         +  400 \sum_{k=0}^{n-1} \left( \frac{4}{5} \right)^k} \\
   B(n) & = & {\displaystyle \left( \frac{4}{5} \right)^n B(0)
         +  400 \sum_{k=0}^{n-1} \left( \frac{4}{5} \right)^k}
\end{eqnarray*}
$$

Now use $B(0) = 0$ and the explicit formula for the geometric sum.

$$
  400 \sum_{k=0}^{n-1} \left( \frac{4}{5} \right)^k
  = 400 \frac{\left( \frac{4}{5} \right)^n - 1}{\frac{4}{5} - 1}
  = 2000 \left( 1 - \left( \frac{4}{5} \right)^n \right)
$$

Alternatively, or as a verification, we can use the ``rsolve`` of ``SymPy``.

In [6]:
n = Sym("n")

n

In [7]:
B = SymFunction("B")

B

In [8]:
equ = Eq(B(n), 0.8*B(n-1) + 400)

B(n) = 0.8*B(n - 1) + 400

In [9]:
init = Dict(B(0) => 0)

Dict{Sym{PyCall.PyObject}, Int64} with 1 entry:
  B(0) => 0

In [10]:
sol = sympy.rsolve(equ, B(n), init)

                   n
2000.0 - 2000.0*0.8 

In doing so, will we ever get rich?  Is there a limit to $B(n)$?

We see that we will never get more than $2000.

# 4. linear programming

A boat rental company must order new boats for next summer. 

There are three types of boats:

|boat type | storage area       | purchase cost | rental price | need for boats|
|  :----:  |   :----:           |    :----:     |   :----:     | :----:        |
|    1     | $f_1=2~{\rm ft}^2$ | $c_1=\$122$   | $p_1=\$5$    | $n_1=20$      |
|    2     | $f_2=3~{\rm ft}^2$ | $c_2=\$130$   | $p_2=\$7$    | $n_2=15$      |
|    3     | $f_3=4~{\rm ft}^2$ | $c_3=\$150$   | $p_3=\$9$    | $n_3=10$      |

Boat $i$ requires $f_i$ square feet to store, costs $c_i$ dollars
to purchase, and can be rented at $p_i$ dollars a trip.

Our constraints are as follows:

* We need at least $n_i$ boats of type $i$,
* but have only $400$ square feet to store the boats, and
* our budget is limited to $\$10,000$.

Determine how many boats of each type we should order.

## defining the problem

In [11]:
boats = DataFrames.DataFrame(
    [
        1 2 122 5 20
        2 3 130 7 15
        3 4 150 9 10
    ],
    ["type", "area", "cost", "price", "need"],
)

Row,type,area,cost,price,need
,Int64,Int64,Int64,Int64,Int64
1,1,2,122,5,20
2,2,3,130,7,15
3,3,4,150,9,10


In [12]:
model = Model(GLPK.Optimizer)

A JuMP Model
├ solver: GLPK
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

In [13]:
@variable(model, x[boats.type] >= 0)

1-dimensional DenseAxisArray{VariableRef,1,...} with index sets:
    Dimension 1, [1, 2, 3]
And data, a 3-element Vector{VariableRef}:
 x[1]
 x[2]
 x[3]

The objective is to maximize the rental price.

In [14]:
@objective(
    model,
    Max,
    sum(boat["price"] * x[boat["type"]] for boat in eachrow(boats)),
)

5 x[1] + 7 x[2] + 9 x[3]

Let us add the constraint on the storage.

In [15]:
storage = @expression(
    model,
    sum(boat["area"] * x[boat["type"]] for boat in eachrow(boats)),
)
@constraint(model, storage <= 400)

2 x[1] + 3 x[2] + 4 x[3] <= 400

The second constraint is on the budget.

In [16]:
budget = @expression(
    model,
    sum(boat["cost"] * x[boat["type"]] for boat in eachrow(boats)),
)
@constraint(model, budget <= 10000)

122 x[1] + 130 x[2] + 150 x[3] <= 10000

And we have a need for each type of boat.

In [17]:
for boat in eachrow(boats)
    @constraint(model, x[boat["type"]] >= boat["need"])
end

To verify whether we have the model defined correctly, we print it.

In [18]:
print(model)

## solving the problem

In [19]:
optimize!(model)
solution_summary(model)

solution_summary(; result = 1, verbose = false)
├ solver_name          : GLPK
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ ├ raw_status         : Solution is optimal
│ └ objective_bound    : Inf
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 5.41600e+02
│ └ dual_objective_value : 5.41600e+02
└ Work counters
  └ solve_time (sec)   : 9.99928e-04

In [20]:
for boat in boats.type
    println("x[",boat, "] = ", value(x[boat]))
end

x[1] = 20.0
x[2] = 15.0
x[3] = 37.4


The above values lead to the value 541.6 for the objective.

# 5. cost benefit analysis

An investment of $10,000 will save us $1,500 each year 
for the coming eight years.

* Use continuous compounding and a discount rate of 6% 
  to compute the present worth of the savings.

* Is the investment worthwhile?

## the present value of the future savings

We compute the present value of the future savings:

In [21]:
s = 1500

1500

In [22]:
r = 0.06

0.06

In [23]:
sum([s*exp(-r*t) for t=1:8])

9247.361701729767

Since the present value is less than $10,000,
the investment is not worthwhile.

# 6. supply and demand

Consider a market with demand $q = 5 + 10/p$ and supply $q = p^2 - 3$.

1. Compute the equilibrium price and the revenue.

2. Suppose the government gives the producer a subsidy
   of $1 per item.  What is the effect of this subsidy?

   Compute the new supply, equilibrium price, and revenue function.

3. Of every dollar the government spends on subsidy,
   how much goes to producer, and how much to the consumer?

## equilibrium and revenue

At the equilibrium, supply equals demand:

$$
   5 + 10/p = p^2 - 3 
$$

Let us use ``NLsolve`` to compute the equilibrium.

In [24]:
"""
    function f!(y, x)

defines y[1] as 5+10/p - p^2 + 3,
where p equals x[1].
"""
function f!(y, x)
    p = x[1]
    y[1] = 5+10/p - p^2 + 3
end

f!

In [25]:
equilibrium = nlsolve(f!, [1.0])

Results of Nonlinear Solver Algorithm
 * Algorithm: Trust-region with dogleg and autoscaling
 * Starting Point: [1.0]
 * Zero: [3.318628217750186]
 * Inf-norm of residuals: 0.000000
 * Iterations: 5
 * Convergence: true
   * |x - x'| < 0.0e+00: false
   * |f(x)| < 1.0e-08: true
 * Function Calls (f): 6
 * Jacobian Calls (df/dx): 6

In [26]:
equilibrium.zero

1-element Vector{Float64}:
 3.318628217750186

So the equilibrium price is $3.32.

Then the revenue is
$$
   R = p \cdot S(p).
$$

In [27]:
S(p) = p^2 - 3

S (generic function with 1 method)

In [28]:
equ_p = equilibrium.zero[1]

3.318628217750186

In [29]:
R = equ_p * S(equ_p)

26.593141088750933

The revenue is $26.59.

The new supply function is $S(p+1)$, or
$$
   S(p+1) = (p+1)^2 - 3.
$$
For the new equilibrium price, use ``NLsolve`` again.   

In [30]:
"""
    function g!(y, x)

defines y[1] as 5+10/p - (p+1)^2 + 3,
where p equals x[1].
"""
function g!(y, x)
    p = x[1]
    y[1] = 5+10/p - (p+1)^2 + 3
end

g!

In [31]:
new_equilibrium = nlsolve(g!, [1.0])

Results of Nonlinear Solver Algorithm
 * Algorithm: Trust-region with dogleg and autoscaling
 * Starting Point: [1.0]
 * Zero: [2.470895516291017]
 * Inf-norm of residuals: 0.000000
 * Iterations: 4
 * Convergence: true
   * |x - x'| < 0.0e+00: false
   * |f(x)| < 1.0e-08: true
 * Function Calls (f): 5
 * Jacobian Calls (df/dx): 5

In [32]:
new_equilibrium.zero

1-element Vector{Float64}:
 2.470895516291017

So the new equilibrium price is $2.47.

In [33]:
new_equ_p = new_equilibrium.zero[1]

2.470895516291017

To compute the new supply, evaluate the supply function at ``new_equ_p`` plus one:

In [34]:
S(new_equ_p+1)

9.047115685009086

The new supply: 9.05.

In [35]:
new_R = new_equ_p*S(new_equ_p+1)

22.354477581455082

The new revenue: $22.35.

The benefit of the consumer:

In [36]:
3.32 - 2.47

0.8499999999999996

so the consumer gets 85 cents for the $1 subsidy.

Of the $1 subsidy, 15 cents goes to the producer.